# Feature Engineering

Creates all predictive features: momentum, volatility, price positioning, liquidity, macro variables, Google Trends, and sector dummies. Features are lagged by one month to avoid look-ahead bias.


In [3]:
from dotenv import load_dotenv; load_dotenv()
import os, re, time, random, requests
import pandas as pd
import numpy as np
from pathlib import Path
from pandas.tseries.offsets import MonthEnd
from pytrends.request import TrendReq
from fredapi import Fred


In [5]:
# Paths & tag
INTERIM_PATH   = Path("../data/interim")
PROCESSED_PATH = Path("../data/processed")
# "1b" for expanded universe
TAG = "1b"  

def _pick_returns_file(tag: str):
    cands = []
    if tag:
        cands.append(INTERIM_PATH / f"returns_monthly_stocks_{tag}.parquet")
    cands.append(INTERIM_PATH / "returns_monthly_stocks.parquet")
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError("No returns_monthly_stocks parquet found.")

p_in = _pick_returns_file(TAG)
df = pd.read_parquet(p_in)
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)
df = df.sort_values(["ticker","date"]).copy()

grouped_df = df.groupby('ticker', group_keys=False)

#  Momentum (t-1 info) 
df['mom_log_3']  = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(3,  min_periods=3).sum())
df['mom_log_6']  = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(6,  min_periods=6).sum())
df['mom_log_12'] = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(12, min_periods=12).sum())

# Volatility 
df['vol_6']  = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(6,  min_periods=6).std())
df['vol_12'] = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(12, min_periods=12).std())

# Beta (36m) 
def rolling_beta(stock, bench, window=36):
    s = stock.shift(1)
    b = bench.shift(1)
    cov_sb = s.rolling(window, min_periods=24).cov(b)
    var_b  = b.rolling(window, min_periods=24).var()
    return cov_sb / var_b

df['beta_36'] = grouped_df.apply(lambda d: rolling_beta(d['l_stock'], d['l_bench']), include_groups=False)

# Reversal & Liquidity
df['rev_1m'] = grouped_df['l_stock'].shift(1)

# Change name to 'liquidity_12m'
df['liquidity_12m'] = grouped_df['volume'].apply(
    lambda s: np.log(s.clip(lower=1)).shift(1).rolling(12, min_periods=6).mean()
)

print(f"Loaded {p_in.name} -> rows={len(df):,}, tickers={df['ticker'].nunique()}")


Loaded returns_monthly_stocks_1b.parquet -> rows=24,129, tickers=100


## Macro Variables

Downloads VIX, Dollar index, US/UK bond yields, exchange rate, and Brent oil from FRED API. Resamples to month-end and lags by one month.


In [8]:
def winsorise_yearly(df, exclude_cols=None, p_low=0.01, p_high=0.99):
    """
    Clips continuous numeric cols per calendar year to [p_low, p_high].
    Skips: ids/date, labels, and binary dummies (0/1).
    """
    out = df.copy()
    out["__year__"] = pd.to_datetime(out["date"]).dt.year

    exclude_cols = set(exclude_cols or [])
    exclude_cols.update({"date", "ticker", "__year__"})

    cand = []
    for c in out.columns:
        if c in exclude_cols:
            continue
        if not pd.api.types.is_numeric_dtype(out[c]):
            continue
        vals = pd.Series(out[c].dropna().unique())
        if len(vals) <= 2 and set(vals.astype(float)).issubset({0.0, 1.0}):
            continue
        cand.append(c)
    if not cand:
        return out.drop(columns="__year__")

    q_low  = out.groupby("__year__")[cand].transform(lambda s: s.quantile(p_low))
    q_high = out.groupby("__year__")[cand].transform(lambda s: s.quantile(p_high))
    out[cand] = out[cand].clip(q_low, q_high)
    return out.drop(columns="__year__")


In [10]:
# Sharpe-like ratio on excess returns 
g_ex   = df.groupby('ticker')['ex_log']
mu_ex12 = g_ex.transform(lambda s: s.shift(1).rolling(12, min_periods=12).mean())
sd_ex12 = g_ex.transform(lambda s: s.shift(1).rolling(12, min_periods=12).std())
df['sharpe_12'] = np.where(sd_ex12 > 0, mu_ex12 / sd_ex12, np.nan)

# 12-month high / SMA / drawdown
px_1      = df.groupby('ticker')['close'].shift(1)
roll_max12 = df.groupby('ticker')['close'].transform(lambda s: s.shift(1).rolling(12, min_periods=12).max())
sma12      = df.groupby('ticker')['close'].transform(lambda s: s.shift(1).rolling(12, min_periods=12).mean())

df['near_high_12'] = px_1 / roll_max12
df['dist_sma_12']  = px_1 / sma12 - 1
df['drawdown_12']  = px_1 / roll_max12 - 1

# Tracking error
df['tracking_error_12'] = (
    df.groupby('ticker')['ex_log']
      .transform(lambda s: s.shift(1).rolling(12, min_periods=12).std())
)

# Up/Down capture (36m)
def up_down_capture_group(g):
    s = g['l_stock'].shift(1)
    b = g['l_bench'].shift(1)
    up_cap   = s.where(b > 0).rolling(36, min_periods=24).mean()
    down_cap = s.where(b < 0).rolling(36, min_periods=24).mean()
    return pd.DataFrame({'up_cap_36': up_cap, 'down_cap_36': down_cap})

cap = (
    df.groupby('ticker', group_keys=False, as_index=False)[['l_stock','l_bench']]
      .apply(up_down_capture_group)
)
df[['up_cap_36','down_cap_36']] = cap[['up_cap_36','down_cap_36']]
df['up_down_cap_ratio_36'] = np.where(df['down_cap_36'].abs() > 1e-8,
                                      df['up_cap_36'] / df['down_cap_36'], np.nan)

# Structural
df['country_UK'] = df['ticker'].astype(str).str.endswith('.L').astype('int8')

# Sector mapping: Load from CSV (all 100 tickers)
sector_csv = INTERIM_PATH / "ticker_sector.csv"
if sector_csv.exists():
    sector_df = pd.read_csv(sector_csv)
    sector_map = dict(zip(sector_df['ticker'].str.strip(), sector_df['sector'].str.strip()))
    df['sector'] = df['ticker'].map(sector_map)
    print(f"Loaded sector mapping from CSV: {len(sector_map)} tickers mapped")
    print(f"   Sectors assigned: {df['sector'].notna().sum()} / {len(df)} rows")
    print(f"   Sector distribution: {df['sector'].value_counts().to_dict()}")
else:
    # Fallback to old hardcoded map (for backward compatibility)
    print("ticker_sector.csv not found, using fallback mapping")
    sector_map = {
        'AAPL':'Tech','JPM':'Financials','XOM':'Energy','PFE':'Healthcare','WMT':'Staples',
        'HSBA.L':'Financials','BP.L':'Energy','TSCO.L':'Staples','AZN.L':'Healthcare','RR.L':'Industrials'
    }
    df['sector'] = df['ticker'].map(sector_map)

# Only make dummies if sector exists for enough names 
if df['sector'].notna().any():
    sector_dummies = pd.get_dummies(df['sector'], prefix='sec', dtype='int8')
    df = pd.concat([df, sector_dummies], axis=1)

# Sector-relative 12m momentum (only where sector available)
if 'mom_log_12' in df.columns and df['sector'].notna().any():
    df['sector_mom12_mean'] = df.groupby(['sector','date'])['mom_log_12'].transform('mean')
    df['sector_rel_mom12']  = df['mom_log_12'] - df['sector_mom12_mean']

# Trends placeholders (kept for later merges)
for c in ['gt_ma3','gt_z12','gt_spike']:
    if c not in df.columns:
        df[c] = np.nan

# Keep schema
id_cols   = ['date','ticker']
base_cols = ['l_stock','l_bench','ex_log','close','volume']
# Build feat_cols: include all sec_* columns dynamically
base_feat_cols = [
    'mom_log_3','mom_log_6','mom_log_12','rev_1m','vol_6','vol_12','sharpe_12',
    'near_high_12','dist_sma_12','drawdown_12',
    'beta_36','tracking_error_12','up_cap_36','down_cap_36','up_down_cap_ratio_36',
    'liquidity_12m',  # <- fixed name
    'country_UK',
    'vix_lvl','dxy_lvl','us10y_lvl','brent_d3m',  # will appear after macro merge
    'gt_ma3','gt_z12','gt_spike',
    'sector_rel_mom12'
]
# Dynamically add all sec_* columns that were created
sector_cols = [c for c in df.columns if c.startswith('sec_')]
feat_cols = base_feat_cols + sector_cols

keep = id_cols + base_cols + [c for c in feat_cols if c in df.columns]
df = df[keep].sort_values(['ticker','date']).copy()

# STEP 3: CLEAN UP HIGH-MISSING FEATURES
# Drop features with >90% NaN 
feat_cols_present = [c for c in df.columns if c not in id_cols + base_cols]
nan_pct = df[feat_cols_present].isnull().mean()
high_missing = [c for c in feat_cols_present if nan_pct[c] > 0.90]

if high_missing:
    print(f"\n[FEATURE CLEANUP] Dropping {len(high_missing)} features with >90% NaN:")
    for c in sorted(high_missing):
        print(f"  - {c}: {nan_pct[c]:.1%} missing")
    df = df.drop(columns=high_missing)
    print(f"  → Remaining features: {len([c for c in df.columns if c not in id_cols + base_cols])}")
else:
    print("\n[FEATURE CLEANUP] No features with >90% NaN to drop")

# Save "core" (pre-macro)
core_path = INTERIM_PATH / (f"features_stocks_core_{TAG}.parquet" if TAG else "features_stocks_core.parquet")
df.to_parquet(core_path, index=False)
print(f"Saved core → {core_path.name} | Rows: {len(df):,} | Cols: {df.shape[1]}")


Loaded sector mapping from CSV: 100 tickers mapped
   Sectors assigned: 24129 / 24129 rows
   Sector distribution: {'Tech': 3913, 'Financials': 3666, 'Industrials': 3203, 'Staples': 3120, 'Healthcare': 2632, 'Discretionary': 2480, 'CommServices': 1774, 'Materials': 1357, 'Energy': 992, 'RealEstate': 496, 'Utilities': 496}

[FEATURE CLEANUP] Dropping 5 features with >90% NaN:
  - down_cap_36: 100.0% missing
  - gt_ma3: 100.0% missing
  - gt_spike: 100.0% missing
  - gt_z12: 100.0% missing
  - up_down_cap_ratio_36: 100.0% missing
  → Remaining features: 27
Saved core → features_stocks_core_1b.parquet | Rows: 24,129 | Cols: 34


## Wikipedia Pageviews

Fetches Wikipedia pageview data as an attention proxy. Features are created but most tickers have insufficient coverage, so these are dropped during feature selection.


In [13]:
# FRED pull (daily to  monthly end, lagged 1m)
fred = Fred(api_key=os.getenv("FRED_API_KEY"))

START, END = "2003-01-01", "2025-12-31"
vix   = fred.get_series('VIXCLS',       observation_start=START, observation_end=END)
brent = fred.get_series('DCOILBRENTEU', observation_start=START, observation_end=END)
dxy   = fred.get_series('DTWEXBGS',     observation_start=START, observation_end=END)
us10y = fred.get_series('DGS10',        observation_start=START, observation_end=END)
dexuk = fred.get_series('DEXUSUK',      observation_start=START, observation_end=END)
uk10y = fred.get_series('IRLTLT01GBM156N', observation_start=START, observation_end=END)  # UK 10-year gilt yield (long-term government bond yield)

daily = (
    pd.concat(
        [vix.rename('vix_lvl'),
         brent.rename('brent'),
         dxy.rename('dxy_lvl'),
         us10y.rename('us10y_lvl'),
         dexuk.rename('dex_usuk'),
         uk10y.rename('uk10y_lvl')], axis=1
    ).sort_index().ffill()
)
daily.index = pd.to_datetime(daily.index)

macro_m = daily.resample('ME').last()
macro_m.index = macro_m.index + MonthEnd(0)
macro_m['brent_d3m'] = np.log(macro_m['brent'].clip(lower=1e-6)).diff(3)

for c in ['vix_lvl','dxy_lvl','us10y_lvl','dex_usuk','brent_d3m','uk10y_lvl']:
    macro_m[c] = macro_m[c].shift(1)

macro_m.index.name = 'date'
macro_m = macro_m[['vix_lvl','dxy_lvl','us10y_lvl','dex_usuk','brent_d3m','uk10y_lvl']].reset_index()

# Save macro cache (shared across runs)
macro_path = INTERIM_PATH / "macro_monthly.parquet"
macro_m.to_parquet(macro_path, index=False)
print("Saved macro:", macro_path.name)

# Merge macro
df = pd.read_parquet(core_path)
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)
df = df.merge(macro_m, on='date', how='left', validate='many_to_one')

# Trim -> QA -> Save BASELINE (v1)
id_cols   = ['date','ticker']
base_cols = ['l_stock','l_bench','ex_log','close','volume']
# Build feat_cols: include all sec_* columns dynamically
base_feat_cols = [
    'mom_log_3','mom_log_6','mom_log_12','rev_1m','vol_6','vol_12','sharpe_12',
    'near_high_12','dist_sma_12','drawdown_12',
    'beta_36','tracking_error_12','up_cap_36','down_cap_36','up_down_cap_ratio_36',
    'liquidity_12m',
    'country_UK',
    'vix_lvl','dxy_lvl','us10y_lvl','uk10y_lvl','dex_usuk','brent_d3m',
    'gt_ma3','gt_z12','gt_spike',
    'sector_rel_mom12'
]
# Dynamically add all sec_* columns that were created
sector_cols = [c for c in df.columns if c.startswith('sec_')]
feat_cols = base_feat_cols + sector_cols
keep = id_cols + base_cols + [c for c in feat_cols if c in df.columns]
df = df.sort_values(['ticker','date'])[keep].copy()

out_v1 = INTERIM_PATH / (f"features_stocks_{TAG}.parquet" if TAG else "features_stocks.parquet")
df.to_parquet(out_v1, index=False)
print(f"Saved v1 (baseline) → {out_v1.name} | rows={len(df):,}, cols={df.shape[1]}")


Saved macro: macro_monthly.parquet
Saved v1 (baseline) → features_stocks_1b.parquet | rows=24,129, cols=40


In [15]:
# GOOGLE TRENDS (UKTRENDS / USTRENDS) 

INTERIM_PATH = Path("../data/interim")
BASE_DIR = INTERIM_PATH / "external_trends"
UK_DIR  = BASE_DIR / "UKTRENDS"
US_DIR  = BASE_DIR / "USTRENDS"
MAP_CSV = INTERIM_PATH / "trends_map.csv"  # ticker,pattern

print("UKTRENDS:", UK_DIR.resolve(), "| exists?", UK_DIR.exists())
print("USTRENDS:", US_DIR.resolve(), "| exists?", US_DIR.exists())
assert UK_DIR.exists(),  f"Missing folder: {UK_DIR}"
assert US_DIR.exists(),  f"Missing folder: {US_DIR}"
assert MAP_CSV.exists(), f"Missing mapping file: {MAP_CSV} (create with columns: ticker,pattern)"

# 1) normalise query header text from Trends CSVs
def normalise_query(s: str) -> str:
    if not isinstance(s, str):
        return s
    t = s.strip().lower()
    t = re.sub(r"\(.*?\)", "", t)             # remove (...) blocks
    t = re.split(r"\s*[:\-–—]\s*", t)[0]       # cut region/category tails
    t = re.sub(r"\s+", " ", t).strip()
    return t

# 2) load mapping
map_df = pd.read_csv(MAP_CSV)
map_df["ticker"]  = map_df["ticker"].astype(str)
map_df["pattern"] = map_df["pattern"].astype(str).str.strip().str.lower()
map_df = map_df[map_df["pattern"] != ""].copy()  # allow blanks to skip (e.g., FERG.L)

MAP_RULES: list[tuple[str, re.Pattern]] = []
for _, r in map_df.iterrows():
    pat = re.escape(r["pattern"])  # treat as literal; put regex in CSV if desired
    MAP_RULES.append((r["ticker"], re.compile(pat)))

def map_query_to_ticker(q: str) -> str | None:
    qn = normalise_query(q)
    for tkr, rgx in MAP_RULES:
        if rgx.search(qn):
            return tkr
    return None

# 3) load all CSVs from both folders

def load_trends_folder(folder: Path) -> list[pd.DataFrame]:
    frames: list[pd.DataFrame] = []
    for f in sorted(folder.glob("*.csv")):
        with open(f, "r", encoding="utf-8") as fh:
            lines = fh.readlines()
        start = next((i for i, L in enumerate(lines) if L.strip().startswith(("Month","Week"))), 0)
        raw = pd.read_csv(f, skiprows=start)
        date_col = raw.columns[0]
        raw = raw.rename(columns={date_col: "date"})
        # show sample headers for mapping assistance
        sample_cols = [c for c in raw.columns if c != "date"][:10]
        print("Sample headers in", f, "→", [normalise_query(c) for c in sample_cols])

        # melt and clean
        long = raw.melt(id_vars="date", var_name="query", value_name="gt_raw")
        long["date"] = pd.to_datetime(long["date"], errors="coerce") + MonthEnd(0)
        long = long.dropna(subset=["date"])
        long["gt_raw"] = pd.to_numeric(long["gt_raw"].replace("<1", 0), errors="coerce")

        # map query -> ticker using CSV mapping
        long["ticker"] = long["query"].apply(map_query_to_ticker)
        unmapped = long.loc[long["ticker"].isna(), "query"].dropna().unique().tolist()
        print(f"Unmapped columns in {f}: {len(unmapped)}")
        if unmapped:
            print("Sample unmapped:", [normalise_query(x) for x in unmapped[:20]])

        long = long.dropna(subset=["ticker"])  # keep only mapped
        frames.append(long[["date","ticker","gt_raw"]])
    return frames

frames = load_trends_folder(UK_DIR) + load_trends_folder(US_DIR)
assert frames, "No Trends CSVs found in UKTRENDS/USTRENDS."

gt = (pd.concat(frames, ignore_index=True)
        .groupby(["ticker","date"], as_index=False)["gt_raw"].mean()
        .sort_values(["ticker","date"]))

print("GT tickers:", gt["ticker"].nunique(), "| GT rows:", len(gt))

# 4) feature engineering (lag 1m)
G = gt.groupby("ticker")["gt_raw"]
gt["gt_ma3"]    = G.transform(lambda s: s.rolling(3,  min_periods=3).mean())
gt["gt_mean12"] = G.transform(lambda s: s.rolling(12, min_periods=12).mean())
gt["gt_std12"]  = G.transform(lambda s: s.rolling(12, min_periods=12).std(ddof=0))
gt["gt_z12"]    = (gt["gt_raw"] - gt["gt_mean12"]) / gt["gt_std12"].replace(0, np.nan)
gt["gt_spike"]  = (gt["gt_z12"] > 2).astype("int8")
for c in ["gt_ma3","gt_z12","gt_spike"]:
    gt[c] = gt.groupby("ticker")[c].shift(1)

gt_feat = gt[["date","ticker","gt_ma3","gt_z12","gt_spike"]].drop_duplicates()

# 5) merge into df  (safe: drop placeholders, merge, coalesce suffixes)
TR_COLS = ["gt_ma3","gt_z12","gt_spike"]

# drop any existing GT columns to avoid _x/_y suffixes on merge
df = df.drop(columns=[c for c in TR_COLS if c in df.columns], errors="ignore")

before = len(df)
df = df.merge(
    gt_feat,
    on=["date","ticker"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_tr")
)

# coalesce any accidental suffix variants back into canonical names
for base in TR_COLS:
    cands = [c for c in (base, f"{base}_tr", f"{base}_x", f"{base}_y") if c in df.columns]
    if cands:
        df[base] = pd.concat([df[c] for c in cands], axis=1).bfill(axis=1).iloc[:, 0]
        for c in cands:
            if c != base:
                df.drop(columns=c, inplace=True, errors="ignore")
    else:
        df[base] = np.nan  # ensure column exists even if nothing merged

print(f"Merged Trends → rows: {before} → {len(df)}")
print("Non-null counts:", {c: int(df[c].notna().sum()) for c in TR_COLS})

# Coverage report
base_set = set(df["ticker"].unique())
feat_set = set(gt_feat["ticker"].unique())
missing  = sorted(base_set - feat_set)
if missing:
    print("Tickers without Trends (first 30):", missing[:30], ("… + more" if len(missing) > 30 else ""))

# Save v2 with tag
try:
    tag = TAG
except NameError:
    tag = "1b"

out_v2 = INTERIM_PATH / (f"features_stocks_v2_{tag}.parquet" if tag else "features_stocks_v2.parquet")
df.to_parquet(out_v2, index=False)
print(f"Saved v2 → {out_v2.name} | rows: {len(df)} | cols: {df.shape[1]}")



UKTRENDS: /Users/umar/Documents/Northeastern/Dissertation/stocks/data/interim/external_trends/UKTRENDS | exists? True
USTRENDS: /Users/umar/Documents/Northeastern/Dissertation/stocks/data/interim/external_trends/USTRENDS | exists? True
Sample headers in ../data/interim/external_trends/UKTRENDS/BATCH1.csv → ['hsbc', 'bp', 'tesco', 'astrazeneca', 'rolls']
Unmapped columns in ../data/interim/external_trends/UKTRENDS/BATCH1.csv: 0
Sample headers in ../data/interim/external_trends/UKTRENDS/BATCH10.csv → ['jd sports', 'sage group', 'smith & nephew', 'smurfit kappa group']
Unmapped columns in ../data/interim/external_trends/UKTRENDS/BATCH10.csv: 0
Sample headers in ../data/interim/external_trends/UKTRENDS/BATCH2.csv → ['shell plc', 'gsk plc', 'british american tobacco bangladesh', 'unilever', 'rio tinto']
Unmapped columns in ../data/interim/external_trends/UKTRENDS/BATCH2.csv: 0
Sample headers in ../data/interim/external_trends/UKTRENDS/BATCH3.csv → ['glencore', 'reckitt', 'lloyds banking gro

In [17]:
# Wikipedia Pageviews 
base_candidates = [
    INTERIM_PATH / (f"features_stocks_v2_{TAG}.parquet" if TAG else "features_stocks_v2.parquet"),
    INTERIM_PATH / (f"features_stocks_{TAG}.parquet"    if TAG else "features_stocks.parquet"),
]
base_path = next((p for p in base_candidates if p.exists()), None)
assert base_path is not None, "No base features file found (run earlier cells first)."
df = pd.read_parquet(base_path)
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)

# Wikipedia mapping for all 100 stocks
WIKI_TITLES = {
    # US Stocks (50)
    "AAPL": "Apple_Inc.",
    "ABBV": "AbbVie",
    "ABT": "Abbott_Laboratories",
    "ACN": "Accenture",
    "ADP": "ADP_(company)",
    "AMAT": "Applied_Materials",
    "AMD": "Advanced_Micro_Devices",
    "AMZN": "Amazon_(company)",
    "AVGO": "Broadcom_Inc.",
    "BA": "Boeing",
    "BAC": "Bank_of_America",
    "BKNG": "Booking_Holdings",
    "CAT": "Caterpillar_Inc.",
    "COST": "Costco",
    "CRM": "Salesforce",
    "CSCO": "Cisco_Systems",
    "CVX": "Chevron_Corporation",
    "DIS": "The_Walt_Disney_Company",
    "GOOGL": "Alphabet_Inc.",
    "GS": "Goldman_Sachs",
    "HD": "The_Home_Depot",
    "HON": "Honeywell",
    "IBM": "IBM",
    "INTC": "Intel",
    "JNJ": "Johnson_&_Johnson",
    "JPM": "JPMorgan_Chase",
    "KO": "The_Coca-Cola_Company",
    "LIN": "Linde_plc",
    "LLY": "Eli_Lilly_and_Company",
    "MA": "Mastercard",
    "MCD": "McDonald's",
    "META": "Meta_Platforms",
    "MRK": "Merck_&_Co.",
    "MSFT": "Microsoft",
    "NFLX": "Netflix",
    "NKE": "Nike_Inc.",
    "NVDA": "Nvidia",
    "ORCL": "Oracle_Corporation",
    "PEP": "PepsiCo",
    "PFE": "Pfizer",
    "PG": "Procter_&_Gamble",
    "PM": "Philip_Morris_International",
    "QCOM": "Qualcomm",
    "TMO": "Thermo_Fisher_Scientific",
    "TXN": "Texas_Instruments",
    "UNH": "UnitedHealth_Group",
    "UPS": "United_Parcel_Service",
    "V": "Visa_Inc.",
    "WMT": "Walmart",
    "XOM": "ExxonMobil",

    # UK Stocks (50) - .L suffix
    "AAL.L": "Anglo_American_plc",
    "ADM.L": "Admiral_Group",
    "AHT.L": "Ashtead_Group",
    "AUTO.L": "Auto_Trader_Group",
    "AV.L": "Aviva",
    "AZN.L": "AstraZeneca",
    "BARC.L": "Barclays",
    "BATS.L": "British_American_Tobacco",
    "BNZL.L": "Bunzl",
    "BP.L": "BP",
    "BT-A.L": "BT_Group",
    "CPG.L": "Compass_Group",
    "DGE.L": "Diageo",
    "EXPN.L": "Experian",
    "FERG.L": "Ferguson_plc",
    "GLEN.L": "Glencore",
    "GSK.L": "GSK_plc",
    "HSBA.L": "HSBC",
    "IAG.L": "International_Airlines_Group",
    "III.L": "3i",
    "JD.L": "JD_Sports_Fashion",
    "LAND.L": "Land_Securities",
    "LGEN.L": "Legal_&_General",
    "LLOY.L": "Lloyds_Banking_Group",
    "MNDI.L": "Mondi",
    "NG.L": "National_Grid_plc",
    "OCDO.L": "Ocado_Group",
    "PRU.L": "Prudential_plc",
    "PSN.L": "Persimmon_plc",
    "PSON.L": "Pearson_plc",
    "REL.L": "RELX",
    "RIO.L": "Rio_Tinto",
    "RKT.L": "Reckitt",
    "RR.L": "Rolls-Royce_Holdings",
    "RTO.L": "Rentokil_Initial",
    "SBRY.L": "Sainsbury's",
    "SDR.L": "Schroders",
    "SGE.L": "Sage_Group",
    "SGRO.L": "Segro",
    "SHEL.L": "Shell_plc",
    "SKG.L": "Smurfit_Kappa_Group",
    "SN.L": "Smith_&_Nephew",
    "SPX.L": "Spirax-Sarco_Engineering",
    "SSE.L": "SSE_plc",
    "STAN.L": "Standard_Chartered",
    "TSCO.L": "Tesco",
    "TW.L": "Taylor_Wimpey",
    "ULVR.L": "Unilever",
    "VOD.L": "Vodafone_Group",
    "WTB.L": "Whitbread",
}

START, END = "20150701", "20251231"
HEADERS = {"User-Agent": "UniDissertationStocks/1.0 (contact@example.com)", "Accept":"application/json"}

def fetch_pageviews_monthly(title: str,
                            start=START, end=END,
                            project="en.wikipedia", access="all-access", agent="user",
                            granularity="monthly", max_retries=4, base_pause=0.6) -> pd.DataFrame:
    url = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
           f"{project}/{access}/{agent}/{title}/{granularity}/{start}/{end}")
    pause = base_pause
    last_err = None
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code in (429, 503, 502, 500):
                raise requests.HTTPError(f"{r.status_code} {r.reason}")
            r.raise_for_status()
            data = r.json().get("items", [])
            if not data:
                return pd.DataFrame(columns=["date","views"])
            out = pd.DataFrame({
                "date": pd.to_datetime([str(it["timestamp"])[:8] for it in data]),
                "views": [it.get("views", np.nan) for it in data]
            })
            out["date"] = out["date"] + MonthEnd(0)
            return out[["date","views"]]
        except Exception as e:
            last_err = e
            if attempt < max_retries - 1:
                sleep_s = pause * (2 ** attempt) + np.random.uniform(0.0, 0.4)
                print(f"[{title}] retry {attempt+1}/{max_retries-1} after: {e}. Sleep {sleep_s:.1f}s")
                time.sleep(sleep_s)
            else:
                print(f"[{title}] FAILED after retries: {e}")
                return pd.DataFrame(columns=["date","views"])

frames = []
for tkr, title in WIKI_TITLES.items():
    pv = fetch_pageviews_monthly(title)
    if pv.empty:
        print(f"[WARN] No pageviews for {tkr} ({title})")
        continue
    pv["ticker"] = tkr
    frames.append(pv[["date","ticker","views"]])
    time.sleep(0.4)

if frames:
    pv_all = (pd.concat(frames, ignore_index=True)
                .sort_values(["ticker","date"])
                .groupby(["ticker","date"], as_index=False)["views"].sum())
    g = pv_all.groupby("ticker")["views"]
    pv_all["pv_ma3"]    = g.transform(lambda s: s.rolling(3,  min_periods=3).mean())
    pv_all["pv_mean12"] = g.transform(lambda s: s.rolling(12, min_periods=12).mean())
    pv_all["pv_std12"]  = g.transform(lambda s: s.rolling(12, min_periods=12).std(ddof=0))
    pv_all["pv_z12"]    = (pv_all["views"] - pv_all["pv_mean12"]) / pv_all["pv_std12"].replace(0, np.nan)
    pv_all["pv_spike"]  = (pv_all["pv_z12"] > 2).astype("int8")
    for c in ["pv_ma3","pv_z12","pv_spike"]:
        pv_all[c] = pv_all.groupby("ticker")[c].shift(1)
    pv_feat = pv_all[["date","ticker","pv_ma3","pv_z12","pv_spike"]].drop_duplicates()
    before = len(df)
    df = df.merge(pv_feat, on=["date","ticker"], how="left", validate="many_to_one")
    print(f"Merged Wikipedia Pageviews → rows: {before} → {len(df)}")
else:
    # ensure columns exist even if skipped
    for c in ["pv_ma3","pv_z12","pv_spike"]:
        if c not in df.columns:
            df[c] = np.nan
    print("Skipped pageviews (no frames).")

# Winsorise v3 (with pv)
exclude_for_v3 = {
    "date","ticker","l_stock","l_bench","ex_log","close","volume",
    "country_UK","sec_Energy","sec_Financials","sec_Healthcare","sec_Industrials","sec_Staples","sec_Tech"
}
df = winsorise_yearly(df, exclude_cols=exclude_for_v3)

# CLEAN UP HIGH-MISSING FEATURES 
id_cols_final = ['date','ticker']
base_cols_final = ['l_stock','l_bench','ex_log','close','volume']
feat_cols_final = [c for c in df.columns if c not in id_cols_final + base_cols_final]
nan_pct_final = df[feat_cols_final].isnull().mean()
high_missing_final = [c for c in feat_cols_final if nan_pct_final[c] > 0.90]

if high_missing_final:
    print(f"\n[FEATURE CLEANUP] Dropping {len(high_missing_final)} features with >90% NaN:")
    for c in sorted(high_missing_final):
        print(f"  - {c}: {nan_pct_final[c]:.1%} missing")
    df = df.drop(columns=high_missing_final)
    print(f"  → Remaining features: {len([c for c in df.columns if c not in id_cols_final + base_cols_final])}")

out_v3 = INTERIM_PATH / (f"features_stocks_v3_{TAG}.parquet" if TAG else "features_stocks_v3.parquet")
df.to_parquet(out_v3, index=False)
print("Saved v3", out_v3.name, "| rows:", len(df), "| cols:", df.shape[1])


Merged Wikipedia Pageviews → rows: 24129 → 24129
Saved v3 features_stocks_v3_1b.parquet | rows: 24129 | cols: 46
